In [1]:
set.seed(405)
options(stringsAsFactors = FALSE)

In [2]:
required_packages <- c("gssr", "dplyr", "forcats", "readr", "tibble")
missing_packages <- required_packages[!vapply(required_packages, requireNamespace, logical(1), quietly = TRUE)]
if (length(missing_packages) > 0) install.packages(missing_packages, repos = "https://cloud.r-project.org")
invisible(lapply(required_packages, library, character.only = TRUE))

Package loaded. To attach the GSS data, type data(gss_all) at the console.
For the panel data and documentation, type e.g. data(gss_panel08_long) and data(gss_panel_doc).
For help on a specific GSS variable, type ?varname at the console.

Warning message:
“package ‘dplyr’ was built under R version 4.4.3”

Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Warning message:
“package ‘readr’ was built under R version 4.4.3”
Warning message:
“package ‘tibble’ was built under R version 4.4.3”


In [3]:
gss_2024 <- gssr::gss_get_yr(2024)
core_vars <- c("id", "confinan", "polviews", "degree", "age", "sex", "income", "region")
raw <- gss_2024 %>% dplyr::select(dplyr::any_of(core_vars))

Fetching: https://gss.norc.org/documents/stata/2024_stata.zip



In [4]:
clean <- raw %>%
  dplyr::mutate(
    confinan = as.character(confinan),
    polviews = suppressWarnings(as.numeric(as.character(polviews))),
    degree = forcats::fct_infreq(as.factor(degree)),
    sex = as.factor(sex),
    income = suppressWarnings(as.numeric(as.character(income))),
    region = as.factor(region)
  ) %>%
  dplyr::mutate(
    confinan_ord = dplyr::case_when(
      confinan %in% c("1", "A GREAT DEAL") ~ 1L,
      confinan %in% c("2", "ONLY SOME") ~ 2L,
      confinan %in% c("3", "HARDLY ANY") ~ 3L,
      TRUE ~ NA_integer_
    ),
    polviews_bin = dplyr::case_when(
      !is.na(polviews) & polviews >= 1 & polviews <= 3 ~ "Liberal",
      !is.na(polviews) & polviews == 4 ~ "Moderate",
      !is.na(polviews) & polviews >= 5 & polviews <= 7 ~ "Conservative",
      TRUE ~ NA_character_
    ),
    polviews_bin = factor(polviews_bin, levels = c("Liberal", "Moderate", "Conservative")),
    age_std = as.numeric(scale(age)),
    income_std = as.numeric(scale(income))
  )

In [5]:
model_tbl <- clean %>%
  dplyr::select(
    id, confinan_ord, polviews, polviews_bin, degree, age, age_std, sex, income, income_std, region
  ) %>%
  dplyr::filter(!is.na(confinan_ord), !is.na(polviews_bin), !is.na(age_std), !is.na(income_std))

In [6]:
encoding_audit <- tibble::tibble(
  metric = c("n_raw", "n_model_tbl", "n_removed", "pct_removed"),
  value = c(
    nrow(raw),
    nrow(model_tbl),
    nrow(raw) - nrow(model_tbl),
    (nrow(raw) - nrow(model_tbl)) / nrow(raw)
  )
)
encoding_audit

metric,value
<chr>,<dbl>
n_raw,3986.0000000
n_model_tbl,2232.0000000
n_removed,1754.0000000
pct_removed,0.4400401


In [7]:
dplyr::count(model_tbl, polviews_bin)
dplyr::count(model_tbl, confinan_ord)

polviews_bin,n
<fct>,<int>
Liberal,720
Moderate,816
Conservative,696


confinan_ord,n
<int>,<int>
1,379
2,1274
3,579


In [8]:
dir.create("../output/derived", recursive = TRUE, showWarnings = FALSE)
readr::write_csv(model_tbl, "../output/derived/gss_2024_model_table.csv")
readr::write_csv(encoding_audit, "../output/derived/encoding_audit.csv")